# Student application 1 — Introduction to AI, ML, and data

Allow 60–90 minutes. Keep the supplied split, seeds, and checks. The matching [PDF brief](01_introduction_and_data_application.pdf) contains the same task guidance and submission checklist.

## Problem statement

An online service receives requests to read or write data. A request is one job sent to the service; latency is the time it takes to finish. The service team wants a warning at arrival if a request will take more than 110 milliseconds (ms). At that moment the finishing time is still unknown. A warning is a prediction, not a measurement of the completed job.

**What you are given.** You are given code that generates 1,200 independent, fictional past requests. For each request, arrival-time information records the amount of data sent (payload, in MB), the number of jobs already waiting (queue), whether reusable data is in fast storage (cache: 1 for a hit, 0 for a miss), and the operation (endpoint: read or write). The completed request supplies its measured latency and therefore its label: 1 for more than 110 ms, otherwise 0. Some payload measurements are missing. The supplied split reserves 720 rows for learning, 240 for checking choices, and 240 for final assessment.

**Concrete example.** Illustration, not an extra dataset row: a request arrives with payload 4 MB, queue 3, cache 0, and endpoint read. These four values can inform a warning. If it later finishes in 125 ms, its recorded label is 1; if it finishes in 90 ms, its label is 0. The eventual time cannot be used to predict itself at arrival.

**The question to answer.** Before building a learned warning model, can we establish trustworthy inputs and measure what a trivial warning rule achieves?

### Your assignment

1. Identify the four measurements available at arrival. Exclude the two completed-outcome fields included as traps in the audit table.
2. Prepare numeric training and validation inputs: fill missing values, scale numeric columns, and encode read/write. Learn all preparation settings from training rows only.
3. Fit a baseline that always predicts the most common training label. On validation rows, measure its accuracy and the proportion of genuinely slow requests it catches.

**Supplied versus your work.** Imports, generated data, fixed row partitions, the preprocessing helper, and check cells are provided. You fill the three marked task cells and write explanations. This application ends with the data audit and baseline; later applications fit richer predictors.

**When you are finished.** Submit the selected feature names, prepared matrix shapes, validation accuracy and slow-request recall, plus the three written answers. Explain whether the baseline can miss slow requests despite high accuracy. Leave test rows unused; a high score is not required to complete the assignment correctly.

## How to work through this application

1. Run the supplied imports and data cells first; they create the variables described below.
2. For each task, read **Goal → Inputs → Steps → Commands → Expected result** before editing.
3. The TODO cell contains a starter outline. Replace each `...` with the requested expression and remove the deliberate `raise NotImplementedError(...)` line when ready.
4. Run the following supplied check and inspect its output. If it fails, correct your task code rather than editing the check.
5. Answer the short interpretation prompts, then restart the kernel and run the completed notebook in order.

**Reading object methods:** `object.method(arguments)` calls an operation on that object. Creating an estimator
sets its options; `fit` learns and stores state; `predict`, `predict_proba`, and `transform` apply learned state.
Training labels belong in `fit`; validation/test labels belong in score calculations.

**Shapes:** `(rows, columns)` describes a table/matrix; `(rows,)` describes a one-dimensional array.
`tr` means training, `va` validation, and `te` test. Raw feature tables use `X`; prepared numeric matrices use `Z`.
Class predictions are labels (0 or 1); event probabilities are numbers between 0 and 1.


**Supplied request-data helper:** `preprocessing()` creates an unfitted transformer. When fitted,
it fills missing numeric values with training medians, subtracts training means and divides by training
standard deviations, and expands `endpoint` into read/write indicator columns. Later `transform`
reuses those choices; an unseen endpoint gets zeros in both indicator columns.

## Variable dictionary

| Name | Meaning |
|---|---|
| `X` | All 1,200 requests, with four arrival-time feature columns: payload (MB), queue (count), cache (0/1), endpoint (read/write). |
| `y` | One label per request: 1 if its eventual latency exceeds 110 ms, otherwise 0. |
| `train, valid, test` | Arrays of row positions. They contain 720, 240, and 240 positions, respectively. |
| `Xtr, Xva, Xte` | Raw feature tables for training, validation, and test. The suffixes tr, va, te identify the partition. |
| `ytr, yva, yte` | Matching target labels, in the same row order as each feature table. |
| `raw` | A table created before task 1: X plus final_latency and final_status, two deliberately illegal prediction-time features. |
| `Ztr, Zva` | Numeric feature matrices AFTER preprocessing, for training and validation. Zva is not a label array or a new split. It will have 240 rows and 5 columns. |
| `prep` | The fitted preprocessing object. It stores training medians, means, standard deviations, and endpoint categories. |

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, log_loss, mean_squared_error, mean_absolute_error)
print('Python', sys.version.split()[0], '| scikit-learn', sklearn.__version__)
plt.rcParams.update({'figure.figsize': (6, 4), 'axes.grid': True, 'grid.alpha': 0.2})

## Supplied data and protocol


## Shared case study: warn before a request exceeds 110 ms
A row is an independent simulated service request. At arrival we know **payload** (MB),
**queue** length, **cache** hit (0/1), and **endpoint** (`read`/`write`). The target is
1 if the eventual latency exceeds 110 ms. Latency itself must never be an input.

The simulator combines linear effects, a queue/cache interaction, and noise with SD 9 ms.
Five percent of payload readings are missing independently. This is teaching data, not evidence
about a real service. We use the same seed and the same 720/240/240 row partitions in sessions
1, 3, 4 and 6. Random splitting is justified by the independent simulation; real repeated users
or future requests would need group or chronological splitting.

`fit` learns from training rows. `transform`, `predict`, and `predict_proba` reuse that state.
All preprocessing is inside each model's pipeline. Validation supports choices; the test is
opened only after those choices are frozen. Since these notebooks share a test partition,
repeated classroom comparisons are illustrations, not independent fresh confirmations.

In [ ]:
# Identical data and row partitions in sessions 1, 3, 4 and 6.
rng = np.random.default_rng(17)
n = 1200
payload = rng.uniform(0.5, 10, n)  # MB, recorded before execution
queue = rng.poisson(3, n)
cache = rng.binomial(1, 0.4, n)
endpoint = rng.choice(['read', 'write'], n, p=[0.7, 0.3])
latency = (35 + 7*payload + 4*queue - 18*cache + 12*(endpoint == 'write')
           + 22*((queue > 4) & (cache == 0)) + rng.normal(0, 9, n))
X = pd.DataFrame({'payload': payload, 'queue': queue, 'cache': cache, 'endpoint': endpoint})
X.loc[rng.random(n) < 0.05, 'payload'] = np.nan  # sensor missingness after outcomes are generated
y = pd.Series((latency > 110).astype(int), name='budget_violation')
train_valid, test = train_test_split(np.arange(n), test_size=0.2, random_state=23, stratify=y)
train, valid = train_test_split(train_valid, test_size=0.25, random_state=23, stratify=y.iloc[train_valid])
assert not (set(train) & set(valid) or set(train) & set(test) or set(valid) & set(test))
assert len(train) + len(valid) + len(test) == n
Xtr, Xva, Xte = X.iloc[train], X.iloc[valid], X.iloc[test]
ytr, yva, yte = y.iloc[train], y.iloc[valid], y.iloc[test]
print('Rows:', len(train), len(valid), len(test), '| Training event rate:', round(ytr.mean(), 3))
def preprocessing():
    return ColumnTransformer([
        ('numeric', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()),
         ['payload', 'queue', 'cache']),
        ('category', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['endpoint'])
    ])
def classification_report_row(y_true, probability, threshold=0.5):
    prediction = (probability >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    return {'accuracy': accuracy_score(y_true, prediction),
            'precision': precision_score(y_true, prediction, zero_division=0),
            'recall': recall_score(y_true, prediction, zero_division=0),
            'F1': f1_score(y_true, prediction, zero_division=0),
            'log_loss': log_loss(y_true, probability, labels=[0, 1]),
            'FP': fp, 'FN': fn, 'cost_per_row': (fp + 5*fn)/len(y_true)}

## Task 1. Identify legal features

**Goal:** Choose features that exist before request execution; do not let the target leak into the inputs.

**Inputs already available:** Use the supplied raw table. Its columns are payload, queue, cache, endpoint, final_latency, and final_status.

### Steps

1. Read each column as an arrival-time measurement or a later outcome. The target depends on final latency, so later outcome columns must be excluded.
2. Create a Python list named legal_features containing exactly the four arrival-time column names as strings.
3. Run the supplied check. Then write one sentence explaining why each excluded column would give the model unavailable information.

### Commands you will use

| Command | What it does |
|---|---|
| `raw.columns` | Returns the table column names; it does not change the table. |
| `raw[legal_features]` | Selects the columns named in a list, retaining all rows. Square brackets here select columns, not a train/test split. |
| `set(legal_features)` | Creates an unordered collection used by the supplied check to compare column names. It does not preprocess data. |

**Expected result:** legal_features is a list of four strings. Neither final_latency nor final_status is present. No model has been fitted yet.

**If you get stuck:** If the check fails, inspect spelling and column meaning. Do not remove the check or include a later outcome merely because it improves accuracy.

**Supplied task setup:** run this cell before editing the starter below.

In [ ]:
raw = X.copy()
raw['final_latency'] = latency
raw['final_status'] = np.where(y == 1, 'slow', 'ok')

In [ ]:
# TODO 1: Identify legal features
# Replace each ... below, then remove the deliberate stop.
raise NotImplementedError("Complete task 1: Identify legal features")

# Choose column names, not their observed values.
legal_features = [...]

**Supplied check and output:** legal_features is a list of four strings. Neither final_latency nor final_status is present. No model has been fitted yet. Assertions stop execution when a stated condition fails; printed scores still need interpretation.

In [ ]:
assert set(legal_features) == {'payload', 'queue', 'cache', 'endpoint'}
assert not {'final_latency', 'final_status'} & set(legal_features)

## Task 2. Fit preprocessing

**Goal:** Create Zva: the validation feature matrix transformed using statistics learned from training only.

**Inputs already available:** Use raw, legal_features, and the supplied train/valid row-position arrays. Create Ztr as the corresponding prepared training matrix.

### Steps

1. Call preprocessing() to create a fresh, unfitted transformer and store it in prep.
2. Select training rows with raw.iloc[train], then legal columns with [legal_features]. Call prep.fit_transform on that table and assign its returned matrix to Ztr.
3. Select validation rows and the same columns. Call prep.transform (not fit_transform) and assign the result to Zva.
4. Run the check and print Ztr.shape and Zva.shape. Explain why four input columns become five prepared columns.

### Commands you will use

| Command | What it does |
|---|---|
| `preprocessing()` | A supplied helper that creates numeric median imputation followed by scaling, plus one-hot encoding of endpoint. Calling it creates an object; it does not yet learn any statistics. |
| `raw.iloc[train]` | Selects rows by integer position. The train array was provided by the split; do not replace it by all rows. |
| `prep.fit_transform(table)` | Learns transformation statistics from table, stores them in prep, and returns the transformed numeric matrix. |
| `prep.transform(table)` | Applies already learned transformations without updating them. This is the correct operation for validation. |
| `matrix.shape` | Reports (number of rows, number of columns). The validation output should be (240, 5): three numeric columns plus two endpoint indicators. |

**Expected result:** Ztr has shape (720, 5); Zva has shape (240, 5). Every entry is finite. Both matrices use exactly the same learned transformation.

**If you get stuck:** Do not overwrite prep by fitting it again on validation. A missing payload is replaced by the training median, not by a validation statistic.

In [ ]:
# TODO 2: Fit preprocessing
# Replace each ... below, then remove the deliberate stop.
raise NotImplementedError("Complete task 2: Fit preprocessing")

prep = preprocessing()
Ztr = prep.fit_transform(...)
Zva = prep.transform(...)

**Supplied check and output:** Ztr has shape (720, 5); Zva has shape (240, 5). Every entry is finite. Both matrices use exactly the same learned transformation. Assertions stop execution when a stated condition fails; printed scores still need interpretation.

In [ ]:
assert Zva.shape[0] == len(valid) and np.isfinite(Zva).all()
print('Prepared validation shape:', Zva.shape)

## Task 3. Measure the baseline

**Goal:** Measure how useful always predicting the training majority class is on new validation requests.

**Inputs already available:** Use Xtr, ytr for fitting and Xva, yva for validation. This dummy model ignores feature values, so it can take the raw tables directly; it does not need Zva.

### Steps

1. Create DummyClassifier(strategy="most_frequent") and store it in baseline.
2. Call baseline.fit(Xtr, ytr) to learn which training label is most common.
3. Call baseline.predict(Xva) and store the labels in prediction.
4. Run the supplied accuracy and recall report. Explain in words how a strong accuracy can coexist with catching no violations. Leave the test unopened.

### Commands you will use

| Command | What it does |
|---|---|
| `DummyClassifier(strategy="most_frequent")` | Constructs a baseline that will predict the most frequent training label for every future row. |
| `baseline.fit(Xtr, ytr)` | Finds the majority label from ytr and stores it in baseline; it returns the fitted object. |
| `baseline.predict(Xva)` | Returns one class label for every validation row without changing fitted state. |
| `accuracy_score(yva, prediction)` | Fraction of validation labels predicted correctly. |
| `recall_score(yva, prediction, zero_division=0)` | Fraction of actual violations caught. The explicit zero convention avoids an undefined numeric result when the relevant denominator is zero. |

**Expected result:** prediction is a length-240 array of labels, not probabilities. The report includes validation accuracy and event recall.

**If you get stuck:** If predict reports that the model is not fitted, run the fit line first. Do not fit using yva to make the baseline look better.

In [ ]:
# TODO 3: Measure the baseline
# Replace each ... below, then remove the deliberate stop.
raise NotImplementedError("Complete task 3: Measure the baseline")

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(..., ...)
prediction = baseline.predict(...)

**Supplied check and output:** prediction is a length-240 array of labels, not probabilities. The report includes validation accuracy and event recall. Assertions stop execution when a stated condition fails; printed scores still need interpretation.

In [ ]:
assert len(prediction) == len(yva)
print('Accuracy:', accuracy_score(yva, prediction))
print('Event recall:', recall_score(yva, prediction, zero_division=0))

## Reading the supplied outputs

The supplied assert statements stop execution if a basic requirement is violated; they do not prove the entire reasoning is correct. np.isfinite checks that transformed values are neither missing nor infinite. Accuracy concerns all rows; recall focuses on violations. No test score is needed in this first data-audit application.

## Written interpretation

1. Explain separately why final_latency and final_status are unavailable at the required prediction time.
2. Describe what happens when endpoint is a category not seen in training. Does successful transformation establish reliable predictions?
3. Propose a split for next month’s requests from repeated users. Distinguish temporal transfer from transfer to entirely new users.

*Write your numbered answers here.*

## Submission
Submit the completed notebook with saved outputs, numbered written answers, and a statement of the fixed or validation-selected settings. Restart the kernel and run all cells. Keep test results out of model selection; application 1 uses validation only.